In [516]:
using DataFrames, Distributions, CSV, Turing, Optim, StatsBase
using StatsFuns: softmax
include("measures.jl")

# Read the vitamin D dataset
data = CSV.read("../../data/vitd/nhanes.csv", DataFrame);

# Look at the first few rows of the data
first(data, 5)

Row,BMXBMI,RIAGENDR,RIDAGEYR,LBXVIDMS,INDFMMPI,DPQ020,SMQ040
,Float64,Float64,Float64,Float64,Float64,Float64,Float64
1,27.8,1.0,62.0,76.1,4.14,0.0,3.0
2,30.8,1.0,53.0,56.5,0.0,0.0,1.0
3,28.8,1.0,78.0,87.5,1.81,0.0,3.0
4,28.0,1.0,22.0,47.2,2.98,0.0,2.0
5,27.6,1.0,46.0,44.5,1.73,0.0,1.0


In [517]:
data[!, :"RIAGENDR"] = data[!, :"RIAGENDR"] .- 1;
data[!, :"DPQ020"] = data[!, :"DPQ020"] .+ 1;

In [518]:
#= Convert categorical data to integers =#

data[!, :"RIAGENDR"] = convert(Array{Int}, data[!, :"RIAGENDR"])
data[!, "DPQ020"] = convert(Array{Int}, data[!, "DPQ020"])
data[!, :"SMQ040"] = convert(Array{Int}, data[!, :"SMQ040"]);

# Get variables
bmi = data[!, :"BMXBMI"]
gender = data[!, :"RIAGENDR"]
age = data[!, :"RIDAGEYR"]
vitd = data[!, :"LBXVIDMS"]
poverty = data[!, :"INDFMMPI"]
depression = data[!, :"DPQ020"]
smoking = data[!, :"SMQ040"];

In [519]:
first(data, 5)

Row,BMXBMI,RIAGENDR,RIDAGEYR,LBXVIDMS,INDFMMPI,DPQ020,SMQ040
,Float64,Int64,Float64,Float64,Float64,Int64,Int64
1,27.8,0,62.0,76.1,4.14,1,3
2,30.8,0,53.0,56.5,0.0,1,1
3,28.8,0,78.0,87.5,1.81,1,3
4,28.0,0,22.0,47.2,2.98,1,2
5,27.6,0,46.0,44.5,1.73,1,1


In [529]:
# Turing Model OPTIM

@model function depression_model(bmi, gender, age, vitd, poverty, depression, smoking)
    n = length(vitd)

    α_age_dist1 ~ Normal(36, 10)

    # α_depression ~ Normal(0, 1.0)

    # β_vitd_depression ~ Normal(0, 1.0)

    # β_bmi_depression ~ Normal(0, 1.0)

    # β_poverty_depression ~ Normal(0, 1.0)

    # β_age_depression ~ Normal(0, 1.0)

    # β_gender_depression ~ Normal(0, 1.0)

    # β_smoking_depression ~ Normal(0, 1.0)


    # Define the linear models
    for i in 1:n
        gender[i] ~ Bernoulli(0.4)
        age_dist1 = Normal(α_age_dist1, 10)
        age_dist2 = Normal(62, 10)
        age[i] ~ MixtureModel([age_dist1, age_dist2], [0.4, 0.6])
        smoking[i] ~ Categorical([0.35, 0.12, 0.53])
    
        bmi[i] ~ LogNormal(3.37, 0.22)
        poverty_dist1 = Normal(2.07895, 2)
        poverty_dist2 = Normal(1.05, 0.001)
        poverty[i] ~ MixtureModel([poverty_dist1, poverty_dist2], [0.5, 0.5])
        vitd[i] ~ LogNormal(4.075, 0.42)

        linear_predictor = α_depression + vitd[i] * β_vitd_depression +
                            bmi[i] * β_bmi_depression + age[i] * β_age_depression +
                            poverty[i] * β_poverty_depression + gender[i] * β_gender_depression +
                           smoking[i] * β_smoking_depression

        probs = softmax([0, linear_predictor, linear_predictor / 2, linear_predictor / 3])

        depression[i] ~ Categorical(probs)

    end
end

depression_model (generic function with 2 methods)

In [530]:
model_optim = depression_model(bmi, gender, age, vitd, poverty, depression, smoking)

DynamicPPL.Model{typeof(depression_model), (:bmi, :gender, :age, :vitd, :poverty, :depression, :smoking), (), (), Tuple{SentinelArrays.ChainedVector{Float64, Vector{Float64}}, Vector{Int64}, SentinelArrays.ChainedVector{Float64, Vector{Float64}}, SentinelArrays.ChainedVector{Float64, Vector{Float64}}, SentinelArrays.ChainedVector{Float64, Vector{Float64}}, Vector{Int64}, Vector{Int64}}, Tuple{}, DynamicPPL.DefaultContext}(depression_model, (bmi = [27.8, 30.8, 28.8, 28.0, 27.6, 24.1, 26.6, 30.3, 35.9, 31.0  …  32.8, 18.8, 32.3, 24.0, 25.1, 53.9, 37.6, 23.4, 32.0, 21.5], gender = [0, 0, 0, 0, 0, 0, 1, 1, 1, 1  …  0, 0, 0, 0, 0, 1, 1, 1, 0, 1], age = [62.0, 53.0, 78.0, 22.0, 46.0, 45.0, 30.0, 69.0, 60.0, 69.0  …  48.0, 63.0, 72.0, 38.0, 62.0, 73.0, 63.0, 72.0, 53.0, 76.0], vitd = [76.1, 56.5, 87.5, 47.2, 44.5, 74.5, 46.3, 103.0, 14.5, 96.5  …  82.6, 39.0, 58.1, 57.8, 73.1, 80.4, 85.2, 91.0, 52.7, 65.5], poverty = [4.14, 0.0, 1.81, 2.98, 1.73, 2.04, 5.0, 0.85, 5.0, 1.43  …  4.94, 0.6, 0.83

In [532]:
map_estimate = optimize(model_optim, MAP(), LBFGS(), Optim.Options(iterations=10000, show_trace=true, show_every=500))

ModeResult with maximized lp of -26410.38
1-element Named Vector{Float64}
A            │ 
─────────────┼───────
:α_age_dist1 │ 34.481

In [494]:
# Turing Model NO BIDIRECTION

@model function depression_model(bmi, gender, age, vitd, poverty, depression, smoking)
    n = length(bmi)

    # Define the linear models
    for i in 1:n
        gender[i] ~ Bernoulli(0.4)
        age_dist1 = Normal(36, 10)
        age_dist2 = Normal(62, 10)
        age[i] ~ MixtureModel([age_dist1, age_dist2], [0.4, 0.6])
        smoking[i] ~ Categorical([0.35, 0.12, 0.53])
    
        bmi[i] ~ LogNormal(3.37, 0.22)
        poverty_dist1 = Normal(2.07895, 2)
        poverty_dist2 = Normal(1.05, 0.001)
        poverty[i] ~ MixtureModel([poverty_dist1, poverty_dist2], [0.5, 0.5])

        vitd[i] ~ LogNormal(4.075, 0.42)

        depression[i] ~ (Normal(0.64 + vitd[i] * 0.000958936 + bmi[i] * 0.00259518 + age[i] * -3.52 * (10^(-5)) + 
        poverty[i] * -0.0743614 + gender[i] * 0.1692 + smoking[i] * -0.112068, 1))

    end
end

depression_model (generic function with 2 methods)

In [495]:
#cont_sampler = HMC(0.01, 10, :bmi, :age, :poverty, :vitd, :depression)
cont_sampler = MH()
disc_sampler = PG(10, :gender, :smoking)
sampler = Gibbs(cont_sampler, disc_sampler)

Gibbs{(:gender, :smoking), Tuple{MH{(), NamedTuple{(), Tuple{}}}, PG{(:gender, :smoking), AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}}}}((MH{(), NamedTuple{(), Tuple{}}}(NamedTuple()), PG{(:gender, :smoking), AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}}(10, AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}(AdvancedPS.resample_systematic, 0.5))))

In [496]:
model = depression_model(bmi, gender, age, vitd, poverty, repeat([missing], length(bmi)), smoking)
#chain = sample(model, sampler, MCMCThreads(), 500, 6)
chain = sample(model, sampler, 5)

Chains MCMC chain (5×1629×1 Array{Float64, 3}):

Iterations        = 1:1:5
Number of chains  = 1
Samples per chain = 5
Wall duration     = 45.22 seconds
Compute duration  = 45.22 seconds
parameters        = depression[1], depression[2], depression[3], depression[4], depression[5], depression[6], depression[7], depression[8], depression[9], depression[10], depression[11], depression[12], depression[13], depression[14], depression[15], depression[16], depression[17], depression[18], depression[19], depression[20], depression[21], depression[22], depression[23], depression[24], depression[25], depression[26], depression[27], depression[28], depression[29], depression[30], depression[31], depression[32], depression[33], depression[34], depression[35], depression[36], depression[37], depression[38], depression[39], depression[40], depression[41], depression[42], depression[43], depression[44], depression[45], depression[46], depression[47], depression[48], depression[49], depression[50], de

In [497]:
depression[1]

0

In [501]:
length(depression)

1628

In [502]:
total = 0
for i in 1:length(depression)
    total += abs(depression[i] - clamp(round(mean(chain["depression[$i]"])), 0, 3))
end
total

994.0

In [503]:
depression_days = Array{Int}(undef, length(depression))

for i in 1:length(depression)
    depression_days[i] = clamp(round(mean(chain["depression[$i]"])), 0, 3)
end

countmap(depression_days)

Dict{Int64, Int64} with 3 entries:
  0 => 985
  2 => 10
  1 => 633